# Instruction Tuning and RLHF with GRPO

In this lecture, we'll be training two models. By the end, one of them will hopefully write positive, uplifting tweets, and the other will write negative, pessimistic ones. We'll start with a small, pretrained language model, and use only ~5K samples of real Twitter data. 

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoModelForSequenceClassification, 
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments
)
from trl import GRPOTrainer, GRPOConfig

## Finetuning
We'll be going back to our old friend GPT-2. Remember that GPT-2 isn't a chatbot and isn't trained to generate any particular kind of text, just English in general. To teach our model to tweet, we need to train it on tweets. 

Technically speaking, what we're doing here is *finetuning*, not *supervised finetuning* (SFT). There's no more supervision here than in regular causal language model training. In other words, we have no labels or prompt-response pairs like we would if we were finetuning the model to be a chatbot. The training objective is ultimately the same as what we'd use in pretraining: we want the most likely next token in a sequence to be the true next token in our training data.

In the following cell, we load our pretrained model and dataset, then tokenize all of our text to get it ready for training. 

In [ ]:
# Load a dataset of tweets for fine tuning.
# We're using the "emotion" subset of the data 
ft_dataset = load_dataset("cardiffnlp/tweet_eval", "emotion", split="train") 
model_name = "gpt2"
block_size = 64 # Max sequence length

# Load our tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token            # Give our model a pad token, same as EOS.

def tokenize_fn(batch):
    return tokenizer(
        [text + tokenizer.eos_token for text in batch["text"]], # Add an EOS token 
        padding=True,                                           # Pad 
        truncation=True,                                        # Truncate sequences that are too long
        max_length=block_size                                   # Set max length
        )

# Tokenize all our tweets
tokenized = ft_dataset.map(tokenize_fn, batched=True, remove_columns=ft_dataset.column_names)


model = AutoModelForCausalLM.from_pretrained(model_name) 
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
sample_prompts = [
    "Just found out that",
    "The best part about",
    "I can't believe",
    "Hot take:",
    "The worst part about",
    "Honestly I think",
    "I'm convinced that",
    "Not gonna lie",
]

def generate_samples(model, prompts, **kwargs):
    model.eval()
    for prompt_text in prompts:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                **kwargs,
            )
        print(f"Prompt: \"{prompt_text}\"\n {tokenizer.decode(ids[0], skip_special_tokens=True)}\n")
        print()

Before we finetune, let's see what base GPT-2 generates for some tweet-style prompts. Since it was trained on general English text (not tweets), we should expect generic, non-tweet-like completions.

In [ ]:
generate_samples(model, sample_prompts, max_new_tokens=64, temperature=0.75)

Finally, we finetune our model. 

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=16,   
    num_train_epochs=1,             
    learning_rate=5e-5,
    warmup_steps=15,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

ft_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

ft_trainer.train()

Now let's see how the model generates after finetuning on tweets. The completions should look much more tweet-like.

In [ ]:
generate_samples(ft_trainer.model, sample_prompts, max_new_tokens=64, temperature=0.75)

## Reinforcement Learning from Human Feedback (RLHF)

We now have a model that generates tweet-like text. But what if we want to control the *sentiment* of its output, making it consistently positive or negative?

This is where **reinforcement learning (RL)** comes in. The idea behind RLHF is simple:

1. **Generate** text from our model (the "policy")
2. **Score** that text using a reward signal (a "reward model" trained on human preference data)
3. **Update** the policy to generate text that scores higher

This is the same core loop behind training models like ChatGPT: generate responses, score them (originally using human preferences, now often using a learned reward model), and update the policy to produce better responses.

### Step 1: Load a Reward Model

We won't be using real human preference data to train a reward model here, but we will still need something to score our model's outputs. We'll use a pretrained **sentiment classifier** trained on tweets. It takes text as input and outputs probabilities for three classes: negative, neutral, and positive. This acts as our reward model, which will serve as a proxy for human judgment.

In [ ]:
REWARD_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
reward_model = AutoModelForSequenceClassification.from_pretrained(REWARD_MODEL_NAME)

In [ ]:
reward_model.config.id2label

In [ ]:
toks = reward_tokenizer(["This tastes terrible!", "This tastes great!"], return_tensors="pt")

with torch.no_grad():
    print(reward_model(**toks).logits.softmax(dim=-1))

### Step 2: Define Reward Functions

GRPO needs a **reward function** that takes a list of prompts and completions and returns a score for each one. We wrap our sentiment classifier in two reward functions:

- `reward_function_positive`: returns the probability of the **positive** class (index 2) -- higher scores for happier text
- `reward_function_negative`: returns the probability of the **negative** class (index 0) -- higher scores for more negative text

We divide the logits by a temperature parameter before the softmax. Lower temperature sharpens the distribution, making the reward more decisive (closer to 0 or 1).

In [ ]:
REWARD_TEMP = .25 

def reward_function_positive(prompts, completions, **kwargs):
    temp = REWARD_TEMP 
    joined = [p + c for p, c in zip(prompts, completions)]
    print(joined[0]) 
    tokenized = reward_tokenizer(joined, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outs = reward_model(**tokenized).logits / temp
    score = (outs.softmax(dim=-1)[:, -1]).flatten().tolist()
    print(f"Mean score: {sum(score)/len(score)}")
    return score

In [ ]:
def reward_function_negative(prompts, completions, **kwargs):
    temp = REWARD_TEMP 
    joined = [p + c for p, c in zip(prompts, completions)]
    print(joined[0]) 
    tokenized = reward_tokenizer(joined, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outs = reward_model(**tokenized).logits / temp
    score = (outs.softmax(dim=-1)[:, 0]).flatten().tolist()
    print(f"Mean score: {sum(score)/len(score)}")
    return score

In [ ]:
reward_function_positive(["This tastes", "This tastes"], [" terrible!", " great!"])

### Step 3: Create a Prompt Dataset

For GRPO, we don't need a dataset of complete texts. Instead, we need a dataset of **prompts**. The model will generate completions for each prompt, and the reward function will score them. Over time, the model learns to complete these prompts in ways that maximize the reward.

We'll use a set of tweet-style openers that invite a variety of completions. I used Claude to generate these quickly. 

In [ ]:
from datasets import Dataset


prompt = {"prompt": [
    "Just found out that",
    "I can't believe",
    "Hot take:",
    "Unpopular opinion:",
    "Why does nobody talk about",
    "The worst part about",
    "The best part about",
    "I'm sorry but",
    "Friendly reminder that",
    "Nobody asked but",
    "Honestly I think",
    "Breaking news:",
    "PSA:",
    "Can we normalize",
    "I will never understand",
    "Thinking about how",
    "It's wild that",
    "Fun fact:",
    "Not me",
    "Tell me why",
    "The fact that",
    "Just realized that",
    "Imagine thinking",
    "I'm convinced that",
    "We need to talk about",
    "I miss when",
    "The internet is",
    "People really out here",
    "Sometimes I think about",
    "You ever just",
    "I don't think people realize",
    "Life is too short to",
    "Today I learned",
    "Shoutout to everyone who",
    "This might be controversial but",
    "I used to think",
    "No one is talking about",
    "Genuinely curious why",
    "In this economy",
    "There are two types of people",
    "If you know you know",
    "I'm starting to think",
    "My hot take is",
    "Normalize not",
    "I promise you",
    "The audacity of",
    "Daily reminder that",
    "Am I the only one who",
    "Controversial take:",
    "Hear me out:",
    "You're not ready for",
    "What if we just",
    "I regret to inform you",
    "Year is 2025 and we still",
    "Just a reminder that",
    "Apparently",
    "The way I just",
    "Me trying to",
    "Society if",
    "I've been thinking and",
    "Okay but why is",
    "New rule:",
    "Can someone explain why",
    "I'm at the point where",
    "Everything reminds me of",
    "Plot twist:",
    "Tired of pretending",
    "It should be illegal to",
    "The older I get the more",
    "Not gonna lie",
    "I would simply",
    "Every day I wake up and",
    "Manifesting",
    "Name one thing better than",
    "If I had a dollar for every",
    "You know what? I'm gonna say it.",
    "That feeling when",
    "I think we can all agree",
    "Actually no,",
    "Just saw someone",
    "What a time to be",
    "No because why does",
    "My therapist would say",
    "It's giving",
    "Very normal behavior to",
    "I simply do not",
    "Turns out that",
    "Another day of",
    "I know it's not that deep but",
    "Respectfully,",
    "The algorithm really said",
    "I'm never going to financially recover from",
    "Me at 3am",
    "Just had an epiphany about",
    "Quick question:",
    "Someone needs to make",
    "We don't deserve",
    "I fear that",
    "Why is no one",
    "Obsessed with the idea of",
]}

dataset = Dataset.from_dict(prompt)

### Step 4: Train with GRPO

**Group Relative Policy Optimization (GRPO)** is an RL algorithm from the DeepSeek team. Here's how it works at each training step:

1. **Generate**: For each prompt, generate $G$ completions from the current policy
2. **Score**: Run each completion through the reward function
3. **Compute advantages**: Normalize rewards *within each group* -- completions that scored above the group mean get positive advantage, below get negative. This is the "Group Relative" part.
4. **Update**: Use a clipped policy gradient objective (similar to PPO) to nudge the model toward high-advantage completions and away from low-advantage ones

You don't need to understand all of this right now. The key thing to understand here is that GRPO doesn't need an absolute sense of "good" -- it only needs to know which completions are *better than others* for the same prompt. This relative comparison makes training more stable.

We use `copy.deepcopy` to ensure each GRPO trainer gets its own independent copy of the fine-tuned model, so the positive and negative trainers don't interfere with each other.

In [ ]:
import copy 

grpo_kwargs = {
    "per_device_train_batch_size": 16, 
    "num_generations": 4,             # How many times we generate per prompt 
    "max_completion_length": 64,      # How many tokens we want to generate
    "save_strategy": "no",          
    "num_train_epochs": 3,
    "learning_rate": 1e-4,
    "temperature": 0.75
}


training_args = GRPOConfig(**grpo_kwargs)

ft_model = copy.deepcopy(ft_trainer.model)

positive_trainer = GRPOTrainer(
    model=copy.deepcopy(ft_model),
    reward_funcs=reward_function_positive,
    args=training_args,
    train_dataset=dataset,
)

positive_trainer.train()

Now let's train the same base model with the **negative** reward function. This model should learn to generate tweets with negative sentiment.

In [ ]:
negative_trainer = GRPOTrainer(
    model=copy.deepcopy(ft_model),
    reward_funcs=reward_function_negative,
    args=training_args,
    train_dataset=dataset,
)

negative_trainer.train()

## Comparing the Models

Let's see how the positive and negative models complete the same prompts. Both started from the same fine-tuned checkpoint: the only difference is which reward function they were trained with.

In [ ]:
gen_kwargs = {
    "max_new_tokens": training_args.max_completion_length,
    "temperature": training_args.temperature,
    "top_p": training_args.top_p,
    "top_k": training_args.top_k,
    "repetition_penalty": training_args.repetition_penalty,
}

print("=== POSITIVE MODEL ===\n")
generate_samples(positive_trainer.model, sample_prompts, **gen_kwargs)

print("=== NEGATIVE MODEL ===\n")
generate_samples(negative_trainer.model, sample_prompts, **gen_kwargs)